In [ ]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================

INPUT_FILE = Path("data/enriched/tanggal_enriched.xlsx")
OUTPUT_DIR = Path("output/second_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["employee_id", "attendance_date"]

CLOCK_COLUMNS = [
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be"
]

# ==========================================
# 2. LOAD DATA
# ==========================================

df = pd.read_excel(INPUT_FILE)

original_count = len(df)

# Preserve original order
df["_original_row"] = range(len(df))

# Normalize business keys
df["employee_id"] = (
    df["employee_id"].astype("string").str.strip()
)

df["attendance_date"] = pd.to_datetime(
    df["attendance_date"], errors="coerce"
).dt.normalize()

# ==========================================
# 3. IDENTIFY EXACT DUPLICATES
# ==========================================

original_columns = [
    c for c in df.columns
    if not c.startswith("_")
]

df["_exact_duplicate"] = df.duplicated(
    subset=original_columns,
    keep="first"
)

# ==========================================
# 4. SCORE CLOCK COMPLETENESS
# ==========================================

def is_filled(series):
    text = series.astype("string").str.strip()

    return (
        series.notna()
        & text.ne("")
        & ~text.str.upper().isin(
            ["NAN", "NONE", "NULL", "NAT"]
        ).fillna(False)
    )

df["_clock_score"] = sum(
    is_filled(df[col]).astype(int)
    for col in CLOCK_COLUMNS
)

# ==========================================
# 5. SCORE ATTENDANCE CODE
# ==========================================

attendance = (
    df["attendance_in"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["_attendance_priority"] = (
    attendance.map({
        "HDR": 2,
        "HOVT": 1
    }).fillna(0)
)

# ==========================================
# 6. SCORE POSTED STATUS
# ==========================================

status = (
    df["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["_status_priority"] = (
    status.map({
        "POSTED": 2,
        "INCOMPLETE": 1
    }).fillna(0)
)

# ==========================================
# 7. SORT BY BUSINESS PRIORITY
# ==========================================

ranked = df.sort_values(
    by=[
        "_exact_duplicate",
        "_clock_score",
        "_attendance_priority",
        "_status_priority",
        "_original_row"
    ],
    ascending=[
        True,
        False,
        False,
        False,
        True
    ],
    kind="stable"
)

# ==========================================
# 8. DEDUPLICATE
# ==========================================

# Rows without valid keys are not automatically merged
valid_keys = ranked[KEY].notna().all(axis=1)

valid = ranked.loc[valid_keys].copy()
invalid = ranked.loc[~valid_keys].copy()

valid["_duplicate_count"] = (
    valid.groupby(KEY)[KEY[0]].transform("size")
)

kept = valid.drop_duplicates(
    subset=KEY,
    keep="first"
).copy()

removed = valid.loc[
    ~valid.index.isin(kept.index)
].copy()

# Restore original row order
cleaned = pd.concat(
    [kept, invalid],
    ignore_index=False
).sort_values("_original_row")

# ==========================================
# 9. AUDIT RESULTS
# ==========================================

print("Original rows:", original_count)
print("Cleaned rows:", len(cleaned))
print("Removed rows:", len(removed))

print(
    "Duplicate groups:",
    (kept["_duplicate_count"] > 1).sum()
)

print(
    "Removed exact duplicates:",
    removed["_exact_duplicate"].sum()
)

# ==========================================
# 10. EXPORT RESULTS
# ==========================================

helper_columns = [
    c for c in cleaned.columns
    if c.startswith("_")
]

with pd.ExcelWriter(
    OUTPUT_DIR / "attendance_deduplication.xlsx"
) as writer:

    cleaned.drop(
        columns=helper_columns
    ).to_excel(
        writer,
        sheet_name="Cleaned Data",
        index=False
    )

    kept.loc[
        kept["_duplicate_count"] > 1
    ].to_excel(
        writer,
        sheet_name="Selected Records",
        index=False
    )

    removed.to_excel(
        writer,
        sheet_name="Removed Records",
        index=False
    )

    invalid.to_excel(
        writer,
        sheet_name="Missing Keys",
        index=False
    )

print("Deduplication completed.")


Original rows: 27774
Cleaned rows: 27711
Removed rows: 63
Duplicate groups: 63
Removed exact duplicates: 0
Deduplication completed.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================


INPUT_FILE = Path("data/enriched/tanggal_enriched.xlsx")
OUTPUT_DIR = Path("output/second_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["employee_id", "attendance_date"]

# Only CURRENT clock fields
CLOCK_COLUMNS = [
    "clock_in_time",
    "clock_out_time"
]

# ==========================================
# 2. LOAD ORIGINAL DATA
# ==========================================

df = pd.read_excel(INPUT_FILE)

original_columns = df.columns.tolist()
original_count = len(df)

required = KEY + CLOCK_COLUMNS + [
    "attendance_in", "status"
]

missing_columns = [
    c for c in required if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

df["_original_row"] = np.arange(len(df))

# ==========================================
# 3. NORMALIZE KEYS
# ==========================================

df["employee_id"] = (
    df["employee_id"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df["attendance_date"] = pd.to_datetime(
    df["attendance_date"],
    errors="coerce"
).dt.normalize()

# ==========================================
# 4. CHECK CURRENT CLOCK COMPLETENESS
# ==========================================

def has_value(series):
    s = series.astype("string").str.strip()

    return (
        series.notna()
        & s.ne("").fillna(False)
        & ~s.str.upper().isin(
            ["NULL", "NONE", "NAN", "NAT", "-"]
        ).fillna(False)
    )

df["_clock_score"] = sum(
    has_value(df[col]).astype(int)
    for col in CLOCK_COLUMNS
)

# Possible scores:
# 0 = both missing
# 1 = one clock available
# 2 = both clocks available

# ==========================================
# 5. ATTENDANCE CODE PRIORITY
# ==========================================

attendance = (
    df["attendance_in"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["_attendance_priority"] = np.select(
    [
        attendance.str.match(
            r"^HDR\b", na=False
        ),
        attendance.str.match(
            r"^HOVT\b", na=False
        )
    ],
    [2, 1],
    default=0
)

# ==========================================
# 6. POSTING STATUS PRIORITY
# ==========================================

status = (
    df["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["_status_priority"] = status.map({
    "POSTED": 2,
    "INCOMPLETE": 1,
    "NOT POSTED": 0
}).fillna(0).astype(int)

# ==========================================
# 7. SEPARATE VALID / MISSING KEYS
# ==========================================

valid_keys = df[KEY].notna().all(axis=1)

valid = df.loc[valid_keys].copy()
missing_keys = df.loc[~valid_keys].copy()

valid["_duplicate_count"] = (
    valid.groupby(KEY)["employee_id"]
    .transform("size")
)

# ==========================================
# 8. RANK DUPLICATES
# ==========================================

ranked = valid.sort_values(
    by=[
        "_clock_score",
        "_attendance_priority",
        "_status_priority",
        "_original_row"
    ],
    ascending=[
        False,
        False,
        False,
        True
    ],
    kind="stable"
)

# ==========================================
# 9. SELECT BEST RECORD
# ==========================================

kept = ranked.drop_duplicates(
    subset=KEY,
    keep="first"
).copy()

removed = ranked.loc[
    ~ranked.index.isin(kept.index)
].copy()

# ==========================================
# 10. CREATE CLEANED DATASET
# ==========================================

cleaned = pd.concat(
    [kept, missing_keys],
    ignore_index=False
).sort_values("_original_row")

# Keep original columns only
final_df = cleaned[original_columns].copy()

# ==========================================
# 11. EXPORT
# ==========================================

final_df.to_excel(
    OUTPUT_DIR / "tanggal_enriched_cleaned.xlsx",
    index=False
)

with pd.ExcelWriter(
    OUTPUT_DIR / "deduplication_audit.xlsx"
) as writer:

    kept.loc[
        kept["_duplicate_count"] > 1
    ].to_excel(
        writer,
        sheet_name="Selected Records",
        index=False
    )

    removed.to_excel(
        writer,
        sheet_name="Removed Records",
        index=False
    )

    missing_keys.to_excel(
        writer,
        sheet_name="Missing Keys",
        index=False
    )

# ==========================================
# 12. SUMMARY
# ==========================================

print("Original rows:", original_count)
print("Final rows:", len(final_df))
print("Removed rows:", len(removed))
print("Missing key rows:", len(missing_keys))

print(
    "Duplicate groups:",
    (kept["_duplicate_count"] > 1).sum()
)

print("\nSaved files:")
print(OUTPUT_DIR / "tanggal_enriched_cleaned.xlsx")
print(OUTPUT_DIR / "deduplication_audit.xlsx")


Original rows: 27774
Final rows: 27711
Removed rows: 63
Missing key rows: 0
Duplicate groups: 63

Saved files:
output\second_reports\tanggal_enriched_cleaned.xlsx
output\second_reports\deduplication_audit.xlsx
